# Phase 4 — Pseudo-Label Generation Pipeline

Runs the full lane-detection → geometry → decision pipeline on the 1,403 MNNIT images
to generate training labels for the decision CNN.

Pipeline stages:
1. **Readiness check** — confirm ONNX weights and images are present
2. **Manual verification** — run geometric logic on 10 samples before committing
3. **Pseudo-label generation** — run the full pipeline at scale
4. **Analysis** — visualise the generated label distribution

In [ ]:
import json
import glob
import os
import sys

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

# Ensure project root is on PYTHONPATH when running the notebook standalone
sys.path.insert(0, str(Path().resolve().parent))

from training.scripts.generate_pseudo_labels import generate_pseudo_labels, PseudoLabelStats

os.makedirs("outputs", exist_ok=True)
print("Imports OK")

## Pipeline Readiness Check

In [ ]:
from app.lane_detection.ufld_model import UFLDLaneDetector
from app.lane_detection.bev_transform import BEVTransform, BEVConfig
from app.decision.geometric_logic import GeometricConfig, compute_geometric_decision

detector = UFLDLaneDetector()
images = sorted(glob.glob("../rgb/rgb_image_*.png"))

print(f"Lane detector ONNX ready : {detector.is_ready()}")
print(f"Images in rgb/           : {len(images)}")

if not detector.is_ready():
    print()
    print("⚠  ONNX weights not found. Pseudo-labeling requires lane_detector.onnx")
    print("   Steps to activate:")
    print("     1. Download UFLDv2 weights from:")
    print("        https://github.com/cfzd/Ultra-Fast-Lane-Detection-v2/releases")
    print("     2. Place .pth at models/lane_detector.pth")
    print("     3. Run: python -m training.scripts.export_onnx --model lane_detector")
    print()
    print("   Continuing with DRY RUN demonstrations below.")

## Geometric Decision Logic — Manual Verification

Run the geometric engine on 10 sample images to validate results **before** pseudo-labeling at scale.

In [ ]:
from app.lane_detection.lane_geometry import LaneGeometryComputer
from app.scene_understanding import SceneContext

geo_config = GeometricConfig.from_yaml("../configs/decision_engine.yaml")
bev_cfg = BEVConfig.from_yaml("../configs/lane_detection.yaml")
bev = BEVTransform(bev_cfg)

# Colour coding per command
COMMAND_COLOURS = {
    "FORWARD": (0.20, 0.75, 0.25),   # green
    "LEFT":    (1.00, 0.70, 0.10),   # amber
    "RIGHT":   (1.00, 0.70, 0.10),   # amber
    "STOP":    (0.90, 0.15, 0.15),   # red
    "N/A":     (0.55, 0.55, 0.55),   # grey
}

sample_paths = images[:10]
scene = SceneContext()  # empty scene (no hazards)

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle("Geometric Engine — 10 Sample Images (before pseudo-labeling)", fontsize=13)

for ax, img_path in zip(axes.flat, sample_paths):
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        ax.axis("off")
        ax.set_title("load error", fontsize=8)
        continue

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    if detector.is_ready():
        detection = detector.predict(img_bgr)
        computer = LaneGeometryComputer(bev, {})
        geometry = computer.compute(detection)
        result = compute_geometric_decision(geometry, scene, geo_config)
        if result:
            cmd = result.command.value
            conf = result.confidence
            offset = result.offset_m
        else:
            cmd, conf, offset = "N/A", 0.0, 0.0
    else:
        # Dry-run: show images with placeholder annotation
        cmd, conf, offset = "N/A (no ONNX)", 0.0, 0.0

    colour = COMMAND_COLOURS.get(cmd.split()[0], (0.5, 0.5, 0.5))
    ax.imshow(img_rgb)
    ax.set_title(f"{cmd}\nconf={conf:.2f}  off={offset:+.2f}m", fontsize=7.5,
                 color=colour, fontweight="bold")
    ax.axis("off")

plt.tight_layout()
plt.savefig("outputs/03_geometric_verification.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved: outputs/03_geometric_verification.png")

## Run Pseudo-Label Generation

In [ ]:
LABELS_PATH = "../data/mnnit/pseudo_labels/labels.jsonl"

if detector.is_ready():
    stats = generate_pseudo_labels(
        image_dir="../rgb",
        output_path=LABELS_PATH,
        config_path="../configs/decision_engine.yaml",
        lane_config_path="../configs/lane_detection.yaml",
        min_lane_confidence=0.85,
        iteration=1,
    )
    stats.print_summary()
else:
    print("Skipping — lane_detector.onnx required for full pipeline.")
    print()
    print("Once the ONNX model is available, re-run this cell.")
    print(f"Labels will be written to: {LABELS_PATH}")

## Pseudo-Label Analysis

In [ ]:
if not Path(LABELS_PATH).exists() or Path(LABELS_PATH).stat().st_size == 0:
    print("No labels generated yet — run the cell above with ONNX model loaded.")
else:
    with open(LABELS_PATH, encoding="utf-8") as fh:
        labels = [json.loads(line) for line in fh if line.strip()]

    commands    = [l["command"] for l in labels]
    confidences = [l["confidence"] for l in labels]
    offsets     = [l["offset_m"] for l in labels]
    curvatures  = [l["curvature_inv_m"] for l in labels]
    cmd_counts  = Counter(commands)

    print(f"Total labels    : {len(labels)}")
    print(f"Coverage        : {len(labels)/1403*100:.1f}% of 1,403 images")
    print(f"Command dist    : {dict(cmd_counts)}")
    print(f"Mean confidence : {np.mean(confidences):.3f}  ±{np.std(confidences):.3f}")

    CMD_ORDER = ["FORWARD", "LEFT", "RIGHT", "STOP"]
    COLOURS   = ["#27ae60", "#f39c12", "#e74c3c", "#8e44ad"]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle("Pseudo-Label Distribution", fontsize=13)

    # --- Plot 1: command bar chart ---
    counts = [cmd_counts.get(c, 0) for c in CMD_ORDER]
    axes[0].bar(CMD_ORDER, counts, color=COLOURS, edgecolor="black", linewidth=0.7)
    axes[0].set_title("Command Distribution")
    axes[0].set_ylabel("Count")
    for i, v in enumerate(counts):
        axes[0].text(i, v + 5, str(v), ha="center", fontsize=9)

    # --- Plot 2: confidence histogram ---
    axes[1].hist(confidences, bins=30, color="#2980b9", edgecolor="white", linewidth=0.5)
    axes[1].axvline(np.mean(confidences), color="red", linestyle="--", label=f"mean={np.mean(confidences):.2f}")
    axes[1].set_title("Confidence Score Distribution")
    axes[1].set_xlabel("Confidence")
    axes[1].set_ylabel("Count")
    axes[1].legend()

    # --- Plot 3: offset vs curvature scatter coloured by command ---
    cmd_colour_map = dict(zip(CMD_ORDER, COLOURS))
    point_colours = [cmd_colour_map.get(c, "#aaaaaa") for c in commands]
    axes[2].scatter(offsets, curvatures, c=point_colours, alpha=0.5, s=12)
    axes[2].axhline(0, color="grey", linewidth=0.5)
    axes[2].axvline(0, color="grey", linewidth=0.5)
    axes[2].set_title("Offset vs Curvature (coloured by command)")
    axes[2].set_xlabel("Lateral Offset (m)")
    axes[2].set_ylabel("Curvature (m⁻¹)")

    plt.tight_layout()
    plt.savefig("outputs/03_label_analysis.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("Saved: outputs/03_label_analysis.png")

In [ ]:
# Show 15 sample labeled images in a 3×5 grid with command overlaid
if not Path(LABELS_PATH).exists() or Path(LABELS_PATH).stat().st_size == 0:
    print("No labels available — run pseudo-label generation first.")
else:
    with open(LABELS_PATH, encoding="utf-8") as fh:
        all_labels = [json.loads(line) for line in fh if line.strip()]

    sample_labels = all_labels[:15]

    CMD_BGR = {
        "FORWARD": (0.20, 0.75, 0.25),
        "LEFT":    (1.00, 0.70, 0.10),
        "RIGHT":   (1.00, 0.70, 0.10),
        "STOP":    (0.90, 0.15, 0.15),
    }

    fig, axes = plt.subplots(3, 5, figsize=(18, 10))
    fig.suptitle("Sample Pseudo-Labeled Images (first 15)", fontsize=13)

    for ax, lbl in zip(axes.flat, sample_labels):
        img_path = os.path.join("..", lbl["image_path"])
        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            ax.axis("off")
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        cmd = lbl["command"]
        colour = CMD_BGR.get(cmd, (0.5, 0.5, 0.5))
        ax.imshow(img_rgb)
        ax.set_title(f"{cmd}\nconf={lbl['confidence']:.2f}", fontsize=8,
                     color=colour, fontweight="bold")
        ax.axis("off")

    plt.tight_layout()
    plt.savefig("outputs/03_sample_labels.png", dpi=100, bbox_inches="tight")
    plt.show()
    print("Saved: outputs/03_sample_labels.png")